# Sprint 2 — Perfilado Demográfico + Análisis Financiero + Acciones Comerciales
**Metodología:** K-Prototypes sobre variables demográficas → análisis financiero a posteriori por segmento → generación automática de perfil y acciones comerciales.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from kmodes.kprototypes import KPrototypes

# Estilo Dark Mode Plotly
pio.templates.default = "plotly_dark"
pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


## 1. Carga de Datos

In [3]:
import os
import glob

# Busqueda automatica del archivo en cualquier maquina
FILENAME = '06-01-2026_Clean.csv'

def encontrar_archivo(nombre):
    # 1. Mismo directorio que el notebook
    nb_dir = os.getcwd()
    ruta_local = os.path.join(nb_dir, nombre)
    if os.path.exists(ruta_local):
        return ruta_local

    # 2. Buscar recursivamente desde el home del usuario
    home = os.path.expanduser('~')
    resultados = glob.glob(os.path.join(home, '**', nombre), recursive=True)
    if resultados:
        return resultados[0]

    # 3. Buscar desde la raiz del disco (ultimo recurso, mas lento)
    import platform
    raiz = 'C:\\' if platform.system() == 'Windows' else '/'
    resultados = glob.glob(os.path.join(raiz, '**', nombre), recursive=True)
    if resultados:
        return resultados[0]

    return None


ruta = encontrar_archivo(FILENAME)

if ruta:
    print(f'Archivo encontrado en: {ruta}')
    df = pd.read_csv(ruta)
    print(f'Shape: {df.shape}')
else:
    raise FileNotFoundError(
        f'No se encontro {FILENAME}.\n'
        'Comprueba que el archivo este en el equipo y vuelve a ejecutar.'
    )

print('\nValores nulos:')
print(df.isna().sum())
df.head()


Archivo encontrado en: c:\Users\HP\OneDrive\Download\Desktop\Simulador\DESAFIO 2\06-01-2026_Clean.csv
Shape: (10987, 18)

Valores nulos:
id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64


,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_pcampaign,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_pcampaign,1
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_pcampaign,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_pcampaign,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_pcampaign,1


## 2. Fase 1 — Perfilado Demográfico (K-Prototypes)
Se entrena el modelo usando **únicamente variables demográficas**: `age`, `job`, `marital`, `education`.

In [4]:
print("--- FASE 1: Perfilado Demográfico ---")

cuanti_d1 = ['age']
categ_d1  = ['job', 'marital', 'education']

df_d1 = df[cuanti_d1 + categ_d1].copy()

scaler_d1 = StandardScaler()
df_d1[cuanti_d1] = scaler_d1.fit_transform(df_d1[cuanti_d1])

X_d1           = df_d1.to_numpy()
cat_indices_d1 = [df_d1.columns.get_loc(col) for col in categ_d1]

kproto_d1 = KPrototypes(n_clusters=4, init='Cao', random_state=42, n_init=3)
df['cluster_demografico'] = kproto_d1.fit_predict(X_d1, categorical=cat_indices_d1)

print("✅ Perfiles demográficos asignados en 'cluster_demografico'.")
print("\nDistribución por segmento:")
print(df['cluster_demografico'].value_counts().sort_index())

--- FASE 1: Perfilado Demográfico ---
✅ Perfiles demográficos asignados en 'cluster_demografico'.

Distribución por segmento:
cluster_demografico
0    1800
1    3220
2    2802
3    3165
Name: count, dtype: int64


In [ ]:
# ============================================================
# 2.1 NOMBRES DESCRIPTIVOS POR CLUSTER
# ============================================================
# Los nombres se generan dinámicamente a partir de los valores
# reales del cluster: etapa de vida (edad) + ocupación modal.
# Si K-Prototypes cambia la asignación al reejecutar, los
# nombres se actualizan solos.

def asignar_nombre_cluster(cluster_id, df, cluster_col='cluster_demografico'):
    """Genera nombre descriptivo según edad media y ocupación modal del cluster."""
    grupo = df[df[cluster_col] == cluster_id]
    age   = grupo['age'].mean()
    job   = grupo['job'].mode()[0]

    if age < 30:
        etapa = "Jóvenes"
    elif age < 45:
        etapa = "Activos"
    elif age < 60:
        etapa = "Consolidados"
    else:
        etapa = "Seniors"

    job_map = {
        'management':    'Directivos',
        'technician':    'Técnicos',
        'blue-collar':   'Operarios',
        'admin.':        'Administrativos',
        'services':      'Servicios',
        'retired':       'Jubilados',
        'self-employed': 'Autónomos',
        'entrepreneur':  'Emprendedores',
        'housemaid':     'Hogar',
        'student':       'Estudiantes',
        'unemployed':    'Desempleados',
        'unknown':       'Mixto'
    }
    ocupacion = job_map.get(job, job.capitalize())
    return f"{etapa} {ocupacion}"

# Construir mapa cluster → nombre
nombre_map = {
    c: asignar_nombre_cluster(c, df)
    for c in sorted(df['cluster_demografico'].unique())
}

# Añadir columna al dataframe
df['nombre_segmento'] = df['cluster_demografico'].map(nombre_map)

print("=== NOMBRES DE SEGMENTO ASIGNADOS ===")
for cid, nombre in nombre_map.items():
    n = (df['cluster_demografico'] == cid).sum()
    print(f"  Cluster {cid} → {nombre}  ({n} clientes)")


## 3. Fase 2 — Análisis Financiero por Segmento
**Sin re-entrenar el modelo.** Se analizan variables financieras sobre los segmentos ya formados.

In [ ]:
# ============================================================
# 3. ANÁLISIS FINANCIERO Y COMERCIAL POR SEGMENTO
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 3.1 Variables financieras por cluster
# ------------------------------------------------------------
cluster_col = 'cluster_demografico'

productos = ['housing', 'loan', 'deposit']

df['total_productos'] = df[productos].sum(axis=1)

resumen_financiero = df.groupby(cluster_col).agg(
    clientes=('age', 'count'),
    edad_media=('age', 'mean'),
    balance_medio=('balance', 'mean'),
    housing_rate=('housing', 'mean'),
    loan_rate=('loan', 'mean'),
    deposit_rate=('deposit', 'mean'),
    productos_promedio=('total_productos', 'mean')
)

# ------------------------------------------------------------
# 3.2 Tasa de conversión del depósito
# ------------------------------------------------------------

resumen_financiero['tasa_conversion_deposit_%'] = (
    resumen_financiero['deposit_rate'] * 100
)

# ------------------------------------------------------------
# 3.3 Índice Relativo de Propensión (IRP)
# ------------------------------------------------------------

promedio_global_productos = df['total_productos'].mean()

resumen_financiero['IRP'] = (
    resumen_financiero['productos_promedio'] / promedio_global_productos
)

# ------------------------------------------------------------
# 3.4 Clasificación comercial
# ------------------------------------------------------------

def clasificar(valor, q1, q2):
    if valor >= q2:
        return "Alta"
    elif valor >= q1:
        return "Media"
    else:
        return "Baja"

q_conv_1 = resumen_financiero['tasa_conversion_deposit_%'].quantile(0.33)
q_conv_2 = resumen_financiero['tasa_conversion_deposit_%'].quantile(0.66)

q_irp_1 = resumen_financiero['IRP'].quantile(0.33)
q_irp_2 = resumen_financiero['IRP'].quantile(0.66)

resumen_financiero['nivel_conversion'] = resumen_financiero['tasa_conversion_deposit_%'].apply(
    lambda x: clasificar(x, q_conv_1, q_conv_2)
)

resumen_financiero['nivel_IRP'] = resumen_financiero['IRP'].apply(
    lambda x: clasificar(x, q_irp_1, q_irp_2)
)

# ------------------------------------------------------------
# 3.5 Estrategia personalizada por cluster
# ------------------------------------------------------------             

def asignar_estrategia(row):
    conv = row['nivel_conversion']
    irp = row['nivel_IRP']

    if conv == "Alta" and irp == "Alta":
        return "Fidelización premium y retención de clientes de alto valor"
    elif conv == "Alta" and irp in ["Media", "Baja"]:
        return "Aumentar vinculación mediante cross-selling gradual"
    elif conv in ["Media", "Baja"] and irp == "Alta":
        return "Activar campañas específicas para convertir alto potencial financiero"
    elif conv == "Media" and irp == "Media":
        return "Desarrollar ofertas personalizadas para elevar conversión y vinculación"
    else:
        return "Reactivación automatizada de bajo coste y campañas digitales simples"

resumen_financiero['estrategia_comercial'] = resumen_financiero.apply(
    asignar_estrategia, axis=1
)

# ------------------------------------------------------------
# 3.6 Tabla ejecutiva final
# ------------------------------------------------------------

tabla_estrategica = resumen_financiero[[
    'clientes',
    'edad_media',
    'balance_medio',
    'tasa_conversion_deposit_%',
    'productos_promedio',
    'IRP',
    'nivel_conversion',
    'nivel_IRP',
    'estrategia_comercial'
]].copy()

tabla_estrategica['edad_media'] = tabla_estrategica['edad_media'].round(1)
tabla_estrategica['balance_medio'] = tabla_estrategica['balance_medio'].round(2)
tabla_estrategica['tasa_conversion_deposit_%'] = tabla_estrategica['tasa_conversion_deposit_%'].round(2)
tabla_estrategica['productos_promedio'] = tabla_estrategica['productos_promedio'].round(2)
tabla_estrategica['IRP'] = tabla_estrategica['IRP'].round(2)

tabla_estrategica

,clientes,edad_media,balance_medio,tasa_conversion_deposit_%,productos_promedio,IRP,nivel_conversion,nivel_IRP,estrategia_comercial
cluster_demografico,,,,,,,,,
0,1800,61.4,2218.31,57.78,0.93,0.86,Alta,Baja,Aumentar vinculación mediante cross-selling gradual
1,3220,30.2,1151.77,50.37,1.18,1.09,Alta,Alta,Fidelización premium y retención de clientes de alto valor
2,2802,36.1,1646.65,49.36,1.08,1.00,Media,Alta,Activar campañas específicas para convertir alto potencial financiero
3,3165,45.6,1448.41,39.30,1.07,0.99,Baja,Media,Reactivación automatizada de bajo coste y campañas digitales simples


In [9]:
# Limpiar columnas de normalización si la celda se re-ejecuta
for _col in ['conv_norm', 'irp_norm', 'clientes_norm', 'score_prioridad', 'ranking_prioridad']:
    if _col in tabla_estrategica.columns:
        tabla_estrategica.drop(columns=_col, inplace=True)

# ============================================================
#  Ranking de Prioridad Comercial
# ============================================================

# Normalización de variables
tabla_estrategica['conv_norm'] = (
    tabla_estrategica['tasa_conversion_deposit_%'] /
    tabla_estrategica['tasa_conversion_deposit_%'].max()
)

tabla_estrategica['irp_norm'] = (
    tabla_estrategica['IRP'] /
    tabla_estrategica['IRP'].max()
)

tabla_estrategica['clientes_norm'] = (
    tabla_estrategica['clientes'] /
    tabla_estrategica['clientes'].max()
)

# Score ponderado de prioridad
# tabla_estrategica['score_prioridad'] = (
#    tabla_estrategica['conv_norm'] * 0.4 +
#    tabla_estrategica['irp_norm'] * 0.4 +
#    tabla_estrategica['clientes_norm'] * 0.2
#)
# Score con ponderación equivalente (1/3 cada variable)     ############################CONSULTAR CON PERFIL CLIENTE Y FINANZAS########################
tabla_estrategica['score_prioridad'] = (
    tabla_estrategica['conv_norm'] +
    tabla_estrategica['irp_norm'] +
    tabla_estrategica['clientes_norm']
) / 3
# Ranking final
tabla_estrategica['ranking_prioridad'] = (
    tabla_estrategica['score_prioridad']
    .rank(ascending=False, method='dense')
    .astype(int)
)

# Ordenar tabla
tabla_estrategica = tabla_estrategica.sort_values(
    by='ranking_prioridad'
)

# Redondear score
tabla_estrategica['score_prioridad'] = (
    tabla_estrategica['score_prioridad']
    .round(3)
)

# Mostrar columnas finales
tabla_ranking = tabla_estrategica[[
    'clientes',
    'tasa_conversion_deposit_%',
    'IRP',
    'score_prioridad',
    'ranking_prioridad',
    'estrategia_comercial'
]]

tabla_ranking

,clientes,tasa_conversion_deposit_%,IRP,score_prioridad,ranking_prioridad,estrategia_comercial
cluster_demografico,,,,,,
1,3220,50.37,1.09,0.957,1,Fidelización premium y retención de clientes de alto valor
2,2802,49.36,1.00,0.881,2,Activar campañas específicas para convertir alto potencial financiero
3,3165,39.30,0.99,0.857,3,Reactivación automatizada de bajo coste y campañas digitales simples
0,1800,57.78,0.86,0.783,4,Aumentar vinculación mediante cross-selling gradual


In [ ]:
# ============================================================
# 4. PERFIL DEMOGRÁFICO POR CLUSTER
# ============================================================
# Responde: ¿quién es cada segmento? (demografía modal + edad media)

perfil_demo = df.groupby('cluster_demografico').agg(
    clientes      = ('age', 'count'),
    edad_media    = ('age', 'mean'),
    edad_mediana  = ('age', 'median'),
    job_top       = ('job',       lambda x: x.value_counts().index[0]),
    marital_top   = ('marital',   lambda x: x.value_counts().index[0]),
    education_top = ('education', lambda x: x.value_counts().index[0])
).round(1)

perfil_demo['descripcion_segmento'] = (
    "Edad media "         + perfil_demo['edad_media'].astype(str)   + " | " +
    "Ocupación: "         + perfil_demo['job_top']                   + " | " +
    "Estado civil: "      + perfil_demo['marital_top']               + " | " +
    "Educación: "         + perfil_demo['education_top']
)

# Añadir nombre descriptivo a perfil_demo
perfil_demo['nombre_segmento'] = pd.Series(nombre_map)

print("=== PERFIL DEMOGRÁFICO POR CLUSTER ===")
perfil_demo[['nombre_segmento','clientes','edad_media','edad_mediana','job_top','marital_top','education_top']]


In [ ]:
# ============================================================
# 5. TABLA UNIFICADA: PERFIL DEMOGRÁFICO + COMPORTAMIENTO FINANCIERO
# ============================================================
# Responde la pregunta completa del Sprint 2.

tabla_final = perfil_demo[
    ['nombre_segmento', 'edad_media', 'job_top', 'marital_top', 'education_top']
].join(
    tabla_estrategica[[
        'balance_medio',
        'tasa_default',
        'housing_rate',
        'loan_rate',
        'tasa_conversion_deposit_%',
        'productos_promedio',
        'IRP',
        'nivel_conversion',
        'nivel_IRP',
        'score_prioridad',
        'ranking_prioridad',
        'estrategia_comercial'
    ]]
).sort_values('ranking_prioridad')

# ── Ranking por conversión de depósito (para Marketing) ──────────────────
tabla_final['ranking_deposit'] = (
    tabla_final['tasa_conversion_deposit_%']
    .rank(ascending=False, method='dense')
    .astype(int)
)

# Formatear rates
for col in ['housing_rate', 'loan_rate']:
    tabla_final[col] = tabla_final[col].astype(str) + '%'

pd.set_option('display.max_colwidth', None)
print("=== PERFIL DEMOGRÁFICO + COMPORTAMIENTO FINANCIERO + ESTRATEGIA ===")
display(tabla_final)

# ── Tabla limpia para Marketing ───────────────────────────────────────────
tabla_marketing = (
    tabla_final[[
        'nombre_segmento',
        'ranking_deposit',
        'edad_media',
        'job_top',
        'marital_top',
        'tasa_conversion_deposit_%',
        'estrategia_comercial'
    ]]
    .sort_values('ranking_deposit')
    .rename(columns={
        'nombre_segmento':             'Segmento',
        'ranking_deposit':             'Prioridad Conversión',
        'edad_media':                  'Edad Media',
        'job_top':                     'Ocupación',
        'marital_top':                 'Estado Civil',
        'tasa_conversion_deposit_%':   'Tasa Depósito (%)',
        'estrategia_comercial':        'Estrategia Recomendada'
    })
)

print("\n=== RANKING PARA MARKETING (por conversión de depósito) ===")
display(tabla_marketing)

# ── Exportar CSVs ─────────────────────────────────────────────────────────
tabla_marketing.to_csv('segmentos_para_marketing.csv', index=False)
df[['cluster_demografico', 'nombre_segmento']].to_csv('clusters_perfil_cliente.csv', index=True)
print("\n✅ Exportado: segmentos_para_marketing.csv")
print("✅ Exportado: clusters_perfil_cliente.csv (para cruce con Finanzas)")


# Conclusiones por Cluster y Estrategias Comerciales

## Cluster 1 – Segmento de máxima prioridad comercial

### Hallazgo

El cluster presentó el mayor score de prioridad comercial, acompañado de un IRP elevado, una tasa de conversión sólida y el mayor volumen de clientes entre los segmentos analizados.

### Interpretación

Este segmento representa la principal oportunidad estratégica para la entidad bancaria, combinando alta capacidad de respuesta comercial, fuerte vinculación financiera y gran impacto potencial en negocio.

### Estrategia comercial

Aplicar estrategias de fidelización premium, fortalecimiento de relación comercial y cross-selling de productos financieros de mayor valor agregado para maximizar la rentabilidad del segmento.

---

## Cluster 2 – Segmento de alto potencial financiero

### Hallazgo

El cluster mostró un IRP elevado y un score de prioridad alto, aunque con una tasa de conversión inferior al segmento líder.

### Interpretación

El segmento posee una intensidad financiera importante y un elevado potencial de crecimiento comercial que aún no ha sido completamente aprovechado.

### Estrategia comercial

Desarrollar campañas personalizadas orientadas a aumentar la conversión efectiva mediante ofertas segmentadas y estrategias de activación financiera.

---

## Cluster 3 – Segmento de prioridad comercial media

### Hallazgo

El cluster presentó niveles intermedios de score de prioridad, IRP y conversión respecto al resto de segmentos.

### Interpretación

El segmento mantiene oportunidades comerciales moderadas, aunque con menor capacidad de generación de valor respecto a los clusters prioritarios.

### Estrategia comercial

Implementar estrategias comerciales de mantenimiento y campañas digitales orientadas a fortalecer gradualmente la vinculación financiera.

---

## Cluster 0 – Segmento de menor prioridad relativa

### Hallazgo

Aunque el cluster presentó la mayor tasa de conversión de depósitos, mostró un menor IRP y un volumen reducido de clientes, disminuyendo su score global de prioridad.

### Interpretación

El segmento responde positivamente a campañas específicas, pero posee menor impacto estratégico global en comparación con los clusters de mayor tamaño y vinculación financiera.

### Estrategia comercial

Aplicar estrategias focalizadas de cross-selling y mantenimiento comercial selectivo, priorizando acciones de bajo coste y alta eficiencia.
